# ENEM 2024 — Pré-processamento (microdados)

## Objetivo
Consolidar variáveis socioeconômicas e notas em um dataset padronizado (nível candidato), com tipagem consistente e categorias ordenadas.

---

## Estratégia de processamento

O pipeline foi estruturado em duas etapas complementares:

### 1) Ingestão e padronização inicial (DuckDB + SQL)
- Leitura eficiente dos arquivos CSV originais
- Seleção de colunas relevantes
- Padronização de nomes
- Exportação para formato Parquet

Essa etapa foi implementada fora do notebook, via scripts SQL e DuckDB, com foco em desempenho e reprodutibilidade.

### 2) Pré-processamento analítico (Pandas)
- Tratamento de valores ausentes
- Recodificação de variáveis socioeconômicas
- Tipagem e ordenação categórica
- Preparação para análise e modelagem

Este notebook cobre **exclusivamente a segunda etapa**, assumindo como entrada os dados já estruturados em Parquet.

---

## Entradas
- Arquivos Parquet derivados dos microdados ENEM 2024

## Saídas
- `DADOS_TRATADOS_2024.parquet`
- `RESULTADOS_TRATADOS_2024.parquet`

## 📌 Observações sobre os dados (ENEM 2024)

Em 2024, o INEP disponibilizou os microdados em dois arquivos separados:

- **Participantes (dados socioeconômicos)**
- **Resultados (notas)**

Além disso:

- Os identificadores foram anonimizados
- Não é possível relacionar diretamente os dois arquivos

Fonte oficial:  
https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem

---

## 📌 Observação sobre o repositório

Os microdados brutos não são versionados neste repositório devido a:

- tamanho elevado dos arquivos
- restrições de redistribuição

Consulte o `README.md` para instruções de obtenção.

### 1) Ambiente e imports 

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote `src/` a partir do diretório do projeto.
ROOT_PATH = Path().resolve().parents[1]  # notebooks/00_preprocessamento -> projeto
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

import re
import numpy as np
import pandas as pd

import seaborn as sns


from src.config import (
    BASE_BRUTA_2024,
    BASE_RESULTADOS_2024,
    DADOS_TRATADOS_2024,
    RESULTADOS_TRATADOS_2024,
)    


from src.preprocessamento.agregacoes import recodificar_microdados_enem
from src.preprocessamento.categorias import (
    ORDEM_ESCOLA, 
    ORDEM_ESTADO_CIVIL,
    ORDEM_FAIXA_ETARIA, 
    ORDEM_LINGUA,    
    ORDEM_OCUPACAO,
    ORDEM_PAIS_ESCOLARIDADE, 
    ORDEM_RACA,
    ORDEM_SAL_MIN, 
    ORDEM_SEXO,


)


pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


### 2) Carregamento dos dados tratados

Os dados são carregados a partir dos arquivos Parquet gerados na etapa de ingestão.

Essa abordagem reduz significativamente o consumo de memória e o tempo de processamento.

### Arquitetura do pipeline

O projeto adota uma arquitetura híbrida:

- **DuckDB + SQL** → ingestão e padronização (dados brutos → Parquet)
- **Pandas** → transformação analítica e modelagem

Essa separação permite:

- melhor desempenho no processamento de grandes volumes
- maior organização do código
- reutilização das bases tratadas em múltiplas análises

In [2]:
df_dados = pd.read_parquet(BASE_BRUTA_2024)
df_resultados = pd.read_parquet(BASE_RESULTADOS_2024)

df_dados.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,municipio,uf,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,banh,quartos,car,moto,gelad,freezer,lv_rp,mcr_ond,asp_po,tv,tv_assin,internet,comptdr,cel,escola
0,5,F,1,1,Porto Alegre,RS,F,F,C,D,4,C,A,C,D,C,B,B,A,B,B,B,D,A,B,B,E,A
1,11,F,1,1,São Luiz Gonzaga,RS,B,C,A,D,2,E,A,B,B,B,B,B,A,A,A,A,B,A,B,A,C,A
2,11,F,1,1,Sarandi,RS,H,F,F,D,5,J,A,C,D,B,B,C,A,B,B,B,D,B,B,A,D,A


In [3]:
df_resultados.head(3)

,uf,escola,municipio,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao
0,CE,2,Aratuba,436.8,377.8,423.4,427.1,1,300
1,SC,4,Tijucas,521.9,601.9,605.5,689.2,0,920
2,PR,None,Rolândia,363,548.4,557.2,456.4,1,480


### 3) Tratamento de valores ausentes
Valores ausentes

Ausências em notas podem refletir não comparecimento/condições específicas — mantidas para decisões posteriores (depende do objetivo).

Ausências em renda_mens são raras e removidas por inviabilizarem a classificação socioeconômica.

Para variáveis categóricas socioeconômicas a exclusão de registros com NaN representaria uma perda significativa de 
informação e potencialmente introduziria viés na amostra.


#### Categoria Não informada(o)

1. **Reconhecimento** que a não-resposta é um dado legítimo
2. **Preservação** a distinção entre diferentes tipos de missing values
3. **Manutenção** a integridade da estrutura categórica ordinal/nominal

Os microdados do ENEM já preveem códigos específicos para "não respondido" ou "não se aplica" em diversas variáveis:

- **Cor/raça:** código 0 ou 6 = "não declarada"
- **Estado civil:** código 0 = "não informado"
- **Escolaridade dos pais:** categoria H = "desconhecido"

Nossa abordagem estende essa lógica de forma consistente para todas as variáveis categóricas, 
incluindo aquelas onde o INEP não prevê explicitamente um código para não-resposta.

In [4]:
df_dados.isna().sum()

faixa_etaria        0
sexo                0
estado_civil        0
cor_raca            0
municipio           0
uf                  0
escolaridade_pai    0
escolaridade_mae    0
ocup_pai            0
ocup_mae            0
n_pessoas_resd      0
sal_min             0
emp_domst           1
banh                1
quartos             0
car                 0
moto                0
gelad               0
freezer             0
lv_rp               0
mcr_ond             1
asp_po              0
tv                  0
tv_assin            2
internet            0
comptdr             0
cel                 0
escola              0
dtype: int64

In [5]:
df_resultados.isna().sum()

uf                    0
escola          2767147
municipio             0
nota_cn         1327963
nota_ch         1164989
nota_lc         1164989
nota_mt         1327963
lingua                0
nota_redacao    1164989
dtype: int64

In [6]:
colunas_criticas = ["emp_domst", "banh", "mcr_ond", "tv_assin"]

df_dados = df_dados.dropna(subset=colunas_criticas).copy()

In [4]:
df_resultados.columns

Index(['uf', 'escola', 'municipio', 'nota_cn', 'nota_ch', 'nota_lc', 'nota_mt',
       'lingua', 'nota_redacao'],
      dtype='object')

### 4) Recodificações, Tipagem e Ordenação
Recodificações socioeconômicas

As respostas do questionário são recodificadas para categorias interpretáveis e/ou variáveis numéricas, mantendo rastreabilidade e consistência longitudinal (2021–2024 quando aplicável).

#### Harmonização da variável escola em 2024

Nos microdados de participantes de 2024, a variável escola apresenta uma categoria adicional: “Não frequentei escola no Ensino Médio”. Essa categoria não aparece nos anos anteriores e possui natureza conceitual distinta das demais, pois não descreve o tipo de instituição frequentada, mas sim uma condição de trajetória escolar.

Como o objetivo desta etapa é manter comparabilidade longitudinal entre 2021 e 2024, a categoria foi agregada a não informada na variável escola.

Essa decisão foi adotada por três razões principais:

Comparabilidade temporal
Nos anos de 2021 a 2023, a variável escola é interpretada no projeto como:

* pública
* privada
* não informada

Manter uma categoria extra apenas em 2024 quebraria a consistência analítica entre os anos.

Diferença conceitual da categoria
A opção “Não frequentei escola no Ensino Médio” não representa um novo tipo de escola, mas uma situação de não frequência escolar. Portanto, sua presença na mesma variável poderia introduzir ambiguidade interpretativa.

Baixa frequência relativa
No arquivo de participantes de 2024, essa categoria corresponde a 13.227 registros, número reduzido em relação ao total da base. Assim, sua agregação em não informada preserva a comparabilidade sem alterar de forma substantiva a distribuição geral da variável.

Dessa forma, a variável escola permanece harmonizada ao longo do pipeline, com a mesma estrutura substantiva usada nos demais anos.

In [16]:
# ----------------------------
# Bens/infra (ENEM 2024)
# ----------------------------
COLS_AB_2024 = [
    "freezer",
    "lv_rp",
    "mcr_ond",
    "asp_po",
    "tv_assin",
    "internet",
]

COLS_ABCD_2024 = [
    "banh",
    "quartos",
    "car",
    "moto",
    "gelad",
    "tv",
]

COLS_ABCDE_2024 = [
    "comptdr",
    "cel",
]


cols_bens_2024 = set(COLS_AB_2024 + COLS_ABCD_2024 + COLS_ABCDE_2024)
faltando_bens = sorted(cols_bens_2024 - set(df_dados.columns))
if faltando_bens:
    print("⚠️ Bens ausentes em df_dados:", faltando_bens)

# ----------------------------
# Recodificação
# ----------------------------

df_dados = recodificar_microdados_enem(
    df_dados,
    schema_escola="dados_2024",
    col_bens_ab=COLS_AB_2024,
    col_bens_abcd=COLS_ABCD_2024,
    col_bens_abcde=COLS_ABCDE_2024,
)
# ----------------------------
# Ordenação das categorias (df_dados)
# ----------------------------
df_dados["cor_raca"] = df_dados["cor_raca"].astype("category").cat.set_categories(ORDEM_RACA, ordered=True)
df_dados["escola"] = df_dados["escola"].astype("category").cat.set_categories(ORDEM_ESCOLA, ordered=True)
df_dados["escolaridade_pai"] = df_dados["escolaridade_pai"].astype("category").cat.set_categories(ORDEM_PAIS_ESCOLARIDADE, ordered=True)
df_dados["escolaridade_mae"] = df_dados["escolaridade_mae"].astype("category").cat.set_categories(ORDEM_PAIS_ESCOLARIDADE, ordered=True)
df_dados["estado_civil"] = df_dados["estado_civil"].astype("category").cat.set_categories(ORDEM_ESTADO_CIVIL, ordered=True)
df_dados["faixa_etaria"] = df_dados["faixa_etaria"].astype("category").cat.set_categories(ORDEM_FAIXA_ETARIA, ordered=True)
df_dados["ocup_pai"] = df_dados["ocup_pai"].astype("category").cat.set_categories(ORDEM_OCUPACAO, ordered=True)
df_dados["ocup_mae"] = df_dados["ocup_mae"].astype("category").cat.set_categories(ORDEM_OCUPACAO, ordered=True)
df_dados["sal_min"] = df_dados["sal_min"].astype("category").cat.set_categories(ORDEM_SAL_MIN, ordered=True)
df_dados["sexo"] = df_dados["sexo"].astype("category").cat.set_categories(ORDEM_SEXO, ordered=True)

# municipio e uf: são categóricas, mas NÃO precisam de ordem
df_dados["municipio"] = df_dados["municipio"].astype("category")
df_dados["uf"] = df_dados["uf"].astype("category")

# colunas int
COLUNAS_INT=["n_pessoas_resd","emp_domst"]+COLS_AB_2024 + COLS_ABCD_2024 + COLS_ABCDE_2024

for coluna in COLUNAS_INT:
    df_dados[coluna] = pd.to_numeric(df_dados[coluna], errors="coerce").astype("Int8")


In [18]:
df_resultados = recodificar_microdados_enem(
    df_resultados,
    schema_escola="resultados_2024",
)
# ----------------------------
# Ordenação das categorias (df_resultados)
# ----------------------------

# colunas categóricas presentes em df_resultados
df_resultados["escola"] = (df_resultados["escola"].astype("category").cat.set_categories(ORDEM_ESCOLA, ordered=True))
df_resultados["lingua"] = (df_resultados["lingua"].astype("category").cat.set_categories(ORDEM_LINGUA, ordered=True))

# municipio e uf: são categóricas, mas NÃO precisam de ordem
df_resultados["municipio"] = df_resultados["municipio"].astype("category")
df_resultados["uf"] = df_resultados["uf"].astype("category")

# colunas float
for coluna in df_resultados.filter(like="nota").columns:
    df_resultados[coluna] = pd.to_numeric(df_resultados[coluna], errors="coerce").astype("float32")


### 5) Criação de Índice de Consumo
Criar um índice sintético com todas as colunas de consumo


In [20]:
CONSUMO_COLS = COLS_AB_2024 + COLS_ABCD_2024 + COLS_ABCDE_2024

consumo = df_dados[CONSUMO_COLS].copy()

# substitui NaN por 0 (se a codificação for consistente)
consumo = consumo.fillna(0)

# normaliza cada coluna para 0-1 usando o máximo observado
consumo_norm = consumo / (consumo.max() + 1e-9)

df_dados["indice_consumo"] = consumo_norm.mean(axis=1).astype("float32")
df_dados["indice_consumo"] = df_dados["indice_consumo"].round(4)

### 6) Dropar colunas de consumo não essenciais

In [22]:
ESSENCIAIS = ["gelad", "lv_rp", "comptdr", "cel", "tv", "indice_consumo"]

# garante que as essenciais existem
missing = [c for c in ESSENCIAIS if c not in df_dados.columns]
if missing:
    raise ValueError(f"Colunas essenciais ausentes: {missing}")

# remove as demais colunas de consumo, mas mantém as essenciais e o índice
cols_consumo_drop = [c for c in CONSUMO_COLS if c not in ESSENCIAIS]
df_dados = df_dados.drop(columns=cols_consumo_drop)


### 7) Conferências finais e exportação
Exportação (Parquet)

O dataset tratado é persistido em Parquet por eficiência (compressão + leitura rápida).

In [24]:
df_dados.to_parquet(DADOS_TRATADOS_2024, index=False)

print("✅ Salvo em:", DADOS_TRATADOS_2024)
print("shape:", df_dados.shape)
print("memória (MB):", df_dados.memory_usage(deep=True).sum() / 1_048_576)

✅ Salvo em: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\dados_tratados_24.parquet
shape: (4332944, 20)
memória (MB): 128.3054962158203


In [25]:
df_resultados.to_parquet(RESULTADOS_TRATADOS_2024, index=False)

print("✅ Salvo em:", RESULTADOS_TRATADOS_2024)
print("shape:", df_resultados.shape)
print("memória (MB):", df_resultados.memory_usage(deep=True).sum() / 1_048_576)

✅ Salvo em: E:\ciencias_dados\projetos\projeto_enem_ml\dados\intermediarios\resultados_tratados_24.parquet
shape: (4332944, 9)
memória (MB): 103.50726699829102
